# IS 698 — Homework-2-UB01976: Tool Deciding Agent
**Evaluating MetaTool Benchmark Evaluation with 3 LLMs**

This notebook has Python code which implements a prompt-based agent that:
1. **Tool Usage Awareness** - Decides **whether** a tool is required for a user query or not.
2. **Tool Selection** - Selects the **correct tool** from 47 available tools.

Evaluated across 3 open-source LLMs via Ollama (llama3.2, mistral, and gemma2).

## 1. Installing the Dependencies and Starting the Ollama Server

We need to start the Ollama server first using **ollama serve** and
then pull the 3 models we will be using for the task:

In [1]:
import subprocess
import time

!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

subprocess.Popen(["ollama", "serve"])
time.sleep(5)

!ollama pull llama3.2:3b
!ollama pull mistral:7b
!ollama pull gemma2:2b

print("All models are installed and ready!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (334 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently i

In [2]:
# Running once to install the required packages for ollama
!pip install ollama pandas scikit-learn tqdm -q

## 2. Random Seed Selection for Reproducibility

In [3]:
import json
import random
import pandas as pd
import numpy as np
from tqdm import tqdm
import ollama
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Three open-source LLMs via Ollama
MODELS = [
    "llama3.2:3b",   # Meta LLaMA 3.2 3B
    "mistral:7b",    # Mistral 7B
    "gemma2:2b",     # Google Gemma 2 2B
]

print("Configuration loaded.")
print(f"Random seed : {RANDOM_SEED}")
print(f"Models      : {MODELS}")

Configuration loaded.
Random seed : 42
Models      : ['llama3.2:3b', 'mistral:7b', 'gemma2:2b']


## 3. Loading Data related to tools (From big_tool_des.json) and Actual data File (all_clean_data.csv)

In [5]:
# Tool data (47 tools) (Load the tools file in the Files section, and as this in Collab file will not be saved after runtime execution and will be lost.)
with open("big_tool_des.json", "r") as f:
    TOOL_DESCRIPTIONS = json.load(f)

TOOL_NAMES = list(TOOL_DESCRIPTIONS.keys())  # 47 tool names

print(f"Loaded {len(TOOL_NAMES)} tool descriptions")
print("Tools:", TOOL_NAMES[:5], "...")

Loaded 47 tool descriptions
Tools: ['FinanceTool', 'ExchangeTool', 'NewsTool', 'PolishTool', 'CharityTool'] ...


In [6]:
# Loading full dataset (Load the full data file in the Files section, and as this is in Collab file will not be saved after runtime execution and will be lost.)
full_df = pd.read_csv("all_clean_data.csv")
print(f"Full dataset shape: {full_df.shape}")
full_df.head(3)

Full dataset shape: (20614, 2)


,Query,Tool
0,Can I find academic research papers on this to...,ResearchHelper
1,Can I find any peer-reviewed papers?,ResearchHelper
2,Can I generate bibtex bibliographies?,ResearchHelper


## 4. Sampling the Dataset

For each of the 47 tools, randomly sample **5 queries** (seed=42 for reproducibility).

In [7]:
SAMPLES_PER_TOOL = 5  # at least 2 required; we are using 5 for better and richer evaluation

sampled_rows = []
for tool in TOOL_NAMES:
    tool_rows = full_df[full_df["Tool"] == tool]
    n = min(SAMPLES_PER_TOOL, len(tool_rows))
    sampled_rows.append(tool_rows.sample(n=n, random_state=RANDOM_SEED))

sampled_df = pd.concat(sampled_rows).reset_index(drop=True)

print(f"Sampled dataset shape : {sampled_df.shape}")
print(f"Queries per tool      : {sampled_df['Tool'].value_counts().describe()}")

# Saving the sampled dataset as it is to be sumbitted.
sampled_df.to_csv("sampled_dataset.csv", index=False)
print("Saved → sampled_dataset.csv")
sampled_df.head()

Sampled dataset shape : (235, 2)
Queries per tool      : count    47.0
mean      5.0
std       0.0
min       5.0
25%       5.0
50%       5.0
75%       5.0
max       5.0
Name: count, dtype: float64
Saved → sampled_dataset.csv


,Query,Tool
0,I want to know the comprehensive and up-to-dat...,FinanceTool
1,Please provide a way to directly obtain the cu...,FinanceTool
2,Can you please provide me with the specific an...,FinanceTool
3,Calculate the drawdowns for the Fidelity Contr...,FinanceTool
4,"Hi, can you help me find the price to earnings...",FinanceTool


## 5. Prompt Design

I am using here a **single prompt** that asks the LLM to:
1. Decide if a tool is needed or not (yes or no)
2. If yes, then in the output give me the exact tool name from the list provided.

In [8]:
def build_tool_list_text(tool_descriptions: dict) -> str:
    """Format all 47 tools into a numbered list for the prompt."""
    lines = []
    for i, (name, desc) in enumerate(tool_descriptions.items(), 1):
        lines.append(f"{i}. {name}: {desc}")
    return "\n".join(lines)


TOOL_LIST_TEXT = build_tool_list_text(TOOL_DESCRIPTIONS)


def build_prompt(query: str) -> str:
    """
    Build the agent prompt for a given user query.

    The prompt instructs the model to:
      - Assess whether any tool is needed
      - If yes, select exactly one tool from the numbered list
      - Return a strict JSON object so we can parse it reliably
    """
    return f"""You are a resourceful intelligent assistant that decides whether a user query requires calling an external tool or not.

You have been provided access to the following 47 tools:

{TOOL_LIST_TEXT}

---
User Query: "{query}"
---

Instructions:
1. Determine whether ANY tool from the list above is needed to answer this query.
2. If a tool IS needed, identify the single most appropriate tool.
3. If there isn't any tool even when required, please return 'None.'
4. Respond ONLY with a valid JSON object in this exact format (no extra text):

{{"tool_needed": "yes" or "no", "tool_name": "<ExactToolName> or null"}}

Rules:
- tool_needed must be exactly "yes" or "no"
- tool_name must be the EXACT tool name from the list (copy it precisely), or null if no tool is needed
- Output ONLY the JSON, no explanation, no markdown fences."""


# Preview the prompt for one query
sample_query = sampled_df.iloc[0]["Query"]
print(f"Sample query: {sample_query}")
print("\n" + "="*60)
print(build_prompt(sample_query)[:800], "...")

Sample query: I want to know the comprehensive and up-to-date financial analysis, including factors such as revenue, expenses, profit margins, cash flow, and balance sheet, specifically for the multinational technology company Amazon.

You are a resourceful intelligent assistant that decides whether a user query requires calling an external tool or not.

You have been provided access to the following 47 tools:

1. FinanceTool: Stay informed with the latest financial updates, real-time insights, and analysis on a wide range of options, stocks, cryptocurrencies, and more.
2. ExchangeTool: Seamlessly convert currencies with our integrated currency conversion tool.
3. NewsTool: Stay connected to global events with our up-to-date news around the world.
4. PolishTool: Elevate your content with our AI-powered tool, which utilizes advanced rewriting techniques to create more human-like expressions and foster creative inspiration.
5. CharityTool: Empower your charitable endeavors by accessing a

## 6. Agent: LLM Inference + Post-processing
To make sure we have proper JSON for processing and do validation against tools available.

In [10]:
import re

def parse_response(raw_text: str, tool_names: list) -> dict:
    """
    Post-process the LLM output into structured fields.

    Strategy:
      1. Try to parse as clean JSON
      2. Fallback: extract tool_needed and tool_name via regex
      3. Validate tool_name against the known 47 tools

    Returns:
        {
          "tool_needed": True/False,
          "predicted_tool": str or None,
          "raw": str
        }
    """
    raw_text = raw_text.strip()

    # Clean JSON parsing
    try:
        # Strip markdown code fences if present
        cleaned = re.sub(r"```(?:json)?\s*", "", raw_text).strip("`").strip()
        data = json.loads(cleaned)
        tool_needed = str(data.get("tool_needed", "")).lower() == "yes"
        predicted_tool = data.get("tool_name", None)
        if predicted_tool in ("null", "None", "", None):
            predicted_tool = None
    except (json.JSONDecodeError, AttributeError):
        # ── Step 2: Regex fallback ────────────────────────────────────────────
        tool_needed_match = re.search(r'tool_needed["\s:]+([\w]+)', raw_text, re.IGNORECASE)
        tool_name_match   = re.search(r'tool_name["\s:]+["\']?([\w&]+)["\']?', raw_text, re.IGNORECASE)

        tool_needed = (tool_needed_match.group(1).lower() == "yes") if tool_needed_match else False
        predicted_tool = tool_name_match.group(1) if tool_name_match else None

    # ── Step 3: Validate against known tool names ─────────────────────────────
    if predicted_tool and predicted_tool not in tool_names:
        lower_map = {t.lower(): t for t in tool_names}
        predicted_tool = lower_map.get(predicted_tool.lower(), None)

    # If tool_needed=True but there is no valid tool found, keep tool_needed=True
    # (Agent claimed a tool was needed but selected an invalid one as there is no valid one)

    return {
        "tool_needed": tool_needed,
        "predicted_tool": predicted_tool,
        "raw": raw_text
    }


def query_llm(model: str, prompt: str, max_retries: int = 2) -> str:
    """Send a prompt to an Ollama-hosted LLM and return raw text."""
    for attempt in range(max_retries + 1):
        try:
            response = ollama.chat(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                options={"temperature": 0.0, "seed": RANDOM_SEED}
            )
            return response["message"]["content"]
        except Exception as e:
            if attempt == max_retries:
                print(f"    [ERROR] {model} failed after {max_retries} retries: {e}")
                return "{\"tool_needed\": \"no\", \"tool_name\": null}"


print("Agent functions are defined.")

Agent functions are defined.


## 7. Run Evaluation

In [11]:
def run_evaluation(model: str, df: pd.DataFrame) -> pd.DataFrame:
    """
    Run the agent on all queries in df and return a results DataFrame.

    For this benchmark all queries in df ARE tool-requiring (label = 1),
    because every row maps to a specific tool.
    """
    print(f"\n{'='*60}")
    print(f"Evaluating model: {model}")
    print(f"{'='*60}")

    results = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=model):
        query      = row["Query"]
        true_tool  = row["Tool"]

        prompt     = build_prompt(query)
        raw_output = query_llm(model, prompt)
        parsed     = parse_response(raw_output, TOOL_NAMES)

        results.append({
            "query"          : query,
            "true_tool"      : true_tool,
            "true_needed"    : 1,            # all sampled queries need a tool
            "pred_needed"    : int(parsed["tool_needed"]),
            "pred_tool"      : parsed["predicted_tool"],
            "correct_select" : int(parsed["predicted_tool"] == true_tool),
            "raw_output"     : parsed["raw"]
        })

    return pd.DataFrame(results)


print("Run Evaluation function is been defined.")

Run Evaluation function is been defined.


In [12]:
# Running all 3 models
# This can take some time as we are running all three models
# Each model processes 235 queries (47 tools × 5 samples).

all_results = {}   # model_name -> results DataFrame

for model in MODELS:
    result_df = run_evaluation(model, sampled_df)
    all_results[model] = result_df
    # Save intermediate results
    safe_name = model.replace(":", "_")
    result_df.to_csv(f"results_{safe_name}.csv", index=False)
    print(f"  Saved → results_{safe_name}.csv")

print("\nAll models evaluated!")


Evaluating model: llama3.2:3b


llama3.2:3b: 100%|██████████| 235/235 [03:45<00:00,  1.04it/s]


  Saved → results_llama3.2_3b.csv

Evaluating model: mistral:7b


mistral:7b: 100%|██████████| 235/235 [05:04<00:00,  1.29s/it]


  Saved → results_mistral_7b.csv

Evaluating model: gemma2:2b


gemma2:2b: 100%|██████████| 235/235 [03:52<00:00,  1.01it/s]

  Saved → results_gemma2_2b.csv

All models evaluated!


## 8. Compute MetaTool Metrics

In [13]:
def compute_metrics(result_df: pd.DataFrame) -> dict:
    """
    Compute the 5 MetaTool evaluation metrics.

    Ground truth: all sampled queries require a tool (true_needed=1).

    Metrics defined in MetaTool paper:
      - Accuracy        : correct tool-needed decisions / total
      - Precision       : TP / (TP + FP)  [tool-use prediction]
      - Recall          : TP / (TP + FN)  [tool-use prediction]
      - F1              : harmonic mean of Precision and Recall
      - CSR             : queries where predicted_tool == true_tool / total
    """
    y_true = result_df["true_needed"].values
    y_pred = result_df["pred_needed"].values

    accuracy  = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall    = recall_score(y_true, y_pred, zero_division=0)
    f1        = f1_score(y_true, y_pred, zero_division=0)
    csr       = result_df["correct_select"].mean()

    return {
        "Accuracy"  : round(accuracy,  4),
        "Precision" : round(precision, 4),
        "Recall"    : round(recall,    4),
        "F1 Score"  : round(f1,        4),
        "CSR"       : round(csr,       4),
    }


# ── Build summary table ───────────────────────────────────────────────────────
metrics_rows = []
for model, result_df in all_results.items():
    metrics = compute_metrics(result_df)
    metrics["Model"] = model
    metrics_rows.append(metrics)

metrics_df = pd.DataFrame(metrics_rows).set_index("Model")[
    ["Accuracy", "Precision", "Recall", "F1 Score", "CSR"]
]

print("\n" + "="*60)
print("MetaTool Evaluation Results")
print("="*60)
print(metrics_df.to_string())
metrics_df.to_csv("evaluation_metrics.csv")
print("\nSaved → evaluation_metrics.csv")


MetaTool Evaluation Results
             Accuracy  Precision  Recall  F1 Score     CSR
Model                                                     
llama3.2:3b    0.9447        1.0  0.9447    0.9716  0.6936
mistral:7b     0.9915        1.0  0.9915    0.9957  0.7362
gemma2:2b      0.9149        1.0  0.9149    0.9556  0.5872

Saved → evaluation_metrics.csv


## 9. Detailed Analysis

In [14]:
# ── Per-tool CSR for each model ───────────────────────────────────────────────
print("Per-Tool Correct Selection Rate (CSR) by Model\n")

per_tool_dfs = []
for model, result_df in all_results.items():
    tool_csr = result_df.groupby("true_tool")["correct_select"].mean().rename(model)
    per_tool_dfs.append(tool_csr)

per_tool_df = pd.concat(per_tool_dfs, axis=1)
per_tool_df["mean_CSR"] = per_tool_df.mean(axis=1)
per_tool_df = per_tool_df.sort_values("mean_CSR", ascending=False)

print(per_tool_df.to_string())

Per-Tool Correct Selection Rate (CSR) by Model

                       llama3.2:3b  mistral:7b  gemma2:2b  mean_CSR
true_tool                                                          
CharityTool                    1.0         1.0        1.0  1.000000
GiftTool                       1.0         1.0        1.0  1.000000
LawTool                        1.0         1.0        1.0  1.000000
ResumeTool                     1.0         1.0        1.0  1.000000
TripTool                       1.0         1.0        1.0  1.000000
MemeTool                       1.0         1.0        1.0  1.000000
BookTool                       1.0         1.0        0.8  0.933333
CompanyInfoTool                0.8         1.0        1.0  0.933333
NotesTool                      1.0         1.0        0.8  0.933333
VideoSummarizeTool             0.8         1.0        1.0  0.933333
ResearchFinder                 0.8         1.0        1.0  0.933333
EarthquakeTool                 1.0         1.0        0.6  0.866667


In [ ]:
# ── Most common prediction errors ─────────────────────────────────────────────
print("Top-10 Confusion Pairs (true_tool → predicted_tool) per Model\n")

for model, result_df in all_results.items():
    errors = result_df[result_df["correct_select"] == 0]
    confusion = (
        errors.groupby(["true_tool", "pred_tool"])
              .size()
              .reset_index(name="count")
              .sort_values("count", ascending=False)
              .head(10)
    )
    print(f"\n{model}")
    print(confusion.to_string(index=False))

Top-10 Confusion Pairs (true_tool → predicted_tool) per Model


llama3.2:3b
          true_tool          pred_tool  count
     TripAdviceTool           TripTool      2
     ResearchHelper     ResearchFinder      2
HousePurchasingTool   HouseRentingTool      2
         CourseTool        FinanceTool      1
         CourseTool    CompanyInfoTool      1
          ChartTool            MapTool      1
  DataRetrievalTool    CompanyInfoTool      1
        PDF&URLTool        FinanceTool      1
        PodcastTool VideoSummarizeTool      1
            JobTool         ResumeTool      1

mistral:7b
            true_tool        pred_tool  count
  HousePurchasingTool HouseRentingTool      2
RestaurantBookingTool            local      2
               Review    ProductSearch      2
       ResearchHelper   ResearchFinder      2
            ChartTool         NASATool      1
              JobTool       ResumeTool      1
           CourseTool         BookTool      1
           CourseTool          JobTool

In [15]:
# ── Sample outputs: correct vs incorrect ──────────────────────────────────────
best_model = metrics_df["CSR"].idxmax()
best_df    = all_results[best_model]

print(f"Sample CORRECT predictions from {best_model}:\n")
correct_samples = best_df[best_df["correct_select"] == 1].head(3)
for _, r in correct_samples.iterrows():
    print(f"  Query : {r['query'][:80]}")
    print(f"  True  : {r['true_tool']}  →  Pred: {r['pred_tool']}  ✓")
    print()

print(f"Sample INCORRECT predictions from {best_model}:\n")
wrong_samples = best_df[best_df["correct_select"] == 0].head(3)
for _, r in wrong_samples.iterrows():
    print(f"  Query : {r['query'][:80]}")
    print(f"  True  : {r['true_tool']}  →  Pred: {r['pred_tool']}  ✗")
    print()

Sample CORRECT predictions from mistral:7b:

  Query : I want to know the comprehensive and up-to-date financial analysis, including fa
  True  : FinanceTool  →  Pred: FinanceTool  ✓

  Query : Please provide a way to directly obtain the current gas fees on the Ethereum net
  True  : FinanceTool  →  Pred: FinanceTool  ✓

  Query : Can you please provide me with the specific and up-to-date figure of the total v
  True  : FinanceTool  →  Pred: FinanceTool  ✓

Sample INCORRECT predictions from mistral:7b:

  Query : What is the value of BTC in USD?
  True  : ExchangeTool  →  Pred: FinanceTool  ✗

  Query : What is the overall sentiment for the EUR/USD currency pair?
  True  : ExchangeTool  →  Pred: FinanceTool  ✗

  Query : Please provide me with a list of the currently trending Show HN projects, displa
  True  : NewsTool  →  Pred: None  ✗



## 10. Final Summary

In [16]:
print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)
print(f"Random Seed           : {RANDOM_SEED}")
print(f"Samples per tool      : {SAMPLES_PER_TOOL}")
print(f"Total evaluation rows : {len(sampled_df)}")
print(f"Number of tools       : {len(TOOL_NAMES)}")
print()
print(metrics_df.to_string())
print()
print(f"Best model by CSR     : {metrics_df['CSR'].idxmax()}  (CSR={metrics_df['CSR'].max():.4f})")
print(f"Best model by F1      : {metrics_df['F1 Score'].idxmax()}  (F1={metrics_df['F1 Score'].max():.4f})")


FINAL RESULTS SUMMARY
Random Seed           : 42
Samples per tool      : 5
Total evaluation rows : 235
Number of tools       : 47

             Accuracy  Precision  Recall  F1 Score     CSR
Model                                                     
llama3.2:3b    0.9447        1.0  0.9447    0.9716  0.6936
mistral:7b     0.9915        1.0  0.9915    0.9957  0.7362
gemma2:2b      0.9149        1.0  0.9149    0.9556  0.5872

Best model by CSR     : mistral:7b  (CSR=0.7362)
Best model by F1      : mistral:7b  (F1=0.9957)
